# 40 · Source-of-edge forensic laboratory

**Research-only / development data.** This notebook aggressively tests feature attribution, group ablations, single-group models, training-only coefficient screening, observable factor-proxy neutralization, tail dependence, temporal and symbol concentration, payoff distributions, paired date-block confidence intervals and a multiple-comparison reality check. **No final historical holdout or prospective ledger access.** Transaction costs = **0** by explicit research assumption. A 2:1 payoff ratio is a diagnostic, not an assumed stop-loss/take-profit fill. All feature-model fits use chronological training data only.

**Interpretation guardrail:** the best-performing model across this large exploratory search is *selection-biased*. Only later genuinely untouched data can confirm an edge. The historical final holdout remains closed.

In [ ]:
from pathlib import Path
import sys, time
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
assert (ROOT/'src').is_dir()
sys.path.insert(0,str(ROOT/'src'))
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.classifier_benchmark import eligible_universe
from stock_ml_dynamic.feature_ablation import development_holdout_split
from stock_ml_dynamic.historical_market import add_historical_market_features
from stock_ml_dynamic.accuracy_features import add_accuracy_features
from stock_ml_dynamic.volume_regime_research import add_volume_regime_features
from stock_ml_dynamic.edge_source_research import run_feature_sources, trade_diagnostics, paired_block_intervals, reality_check
from stock_ml_dynamic.payoff_search import payoff_stats
OUT=ROOT/'artifacts/edge_sources/1d';OUT.mkdir(parents=True,exist_ok=True)
PRIOR=ROOT/'artifacts/edge_discovery/1d'
master,latest,bundle,manifest=load_master(ROOT/'artifacts/master_tables/1d')
print('Master:',master.shape,'symbols:',master.symbol.nunique())

## Experiment 1 — Reconstruct the development-only research universe

In [ ]:
eligible,eligibility=eligible_universe(master)
development,cutoff=development_holdout_split(eligible,.20)
expanded,historical=add_historical_market_features(development)
expanded,technical=add_accuracy_features(expanded)
expanded,volume=add_volume_regime_features(expanded)
features=list(dict.fromkeys(historical+technical+volume))
assert len(development)<len(eligible)
print('Development:',len(expanded),'engineered features:',len(features),'holdout boundary:',cutoff)
print('Features by source:',{'historical':len(historical),'technical':len(technical),'volume':len(volume)})

## Experiment 2 — Frozen Notebook 39 picks: verify files and baseline

In [ ]:
required=['trades.csv','metrics.csv','daily.csv']
missing=[p for p in required if not (PRIOR/p).exists()]
if missing:print('NOTE: Notebook 39 saved results missing:',missing);print('Run Notebook 39 to enable frozen-pick diagnostics. Feature experiments below can still run.')
else:
    frozen_trades=pd.read_csv(PRIOR/'trades.csv',parse_dates=['bar_timestamp'])
    frozen_metrics=pd.read_csv(PRIOR/'metrics.csv')
    frozen_daily=pd.read_csv(PRIOR/'daily.csv',parse_dates=['bar_timestamp'])
    print('Frozen Notebook 39 trades:',len(frozen_trades),'models:',frozen_trades.model.nunique())
    display(frozen_metrics.query('model=="ridge_a100"').round(6))

## Experiments 3–9 — Frozen-trade forensics: tails, winners, losers, years, symbols, drawdowns, 2:1 payoff

In [ ]:
if not missing:
    forensic=trade_diagnostics(frozen_trades,model='ridge_a100',k=5)
    for key in ('by_year','concentration','top_winners','bottom_losers'):
        print(key);display(forensic[key].head(30).round(6))
    print('Top contributing symbols');display(forensic['by_symbol'].head(25).round(6))
    print('Worst contributing symbols');display(forensic['by_symbol'].tail(25).round(6))
    for key,df in forensic.items():df.to_csv(OUT/f'frozen_{key}.csv',index=False)
    print('Per-fold maximum drawdown');display(forensic['by_date'].groupby('fold').drawdown.min())
    print('Payoff ratios by fold:');display(frozen_metrics.query('model=="ridge_a100" and k==5')[['fold','win_rate','avg_win','avg_loss','payoff_ratio','profit_factor','expectancy']].round(5))

## Experiment 10 — Calendar-month stability and exceptional winner concentration

In [ ]:
if not missing:
    month=forensic['by_month'];display(month.sort_values('mean').head(15).round(5));display(month.sort_values('mean').tail(15).round(5))
    fig,ax=plt.subplots(figsize=(12,4));d=forensic['by_date'];
    for fold,g in d.groupby('fold'):ax.plot(g.bar_timestamp,g.equity,label=f'Fold {fold}')
    ax.set(title='Ridge α100 top-five gross equity (each fold starts at 1)',ylabel='Compounded gross value');ax.legend();plt.tight_layout();plt.show()

## Experiments 11–30+ — Feature-source ablation, isolated groups, top coefficients, observable factor neutralization

In [ ]:
SEED=42;TRAIN_CAP=60000
start=time.perf_counter()
res=run_feature_sources(expanded,features,cap=TRAIN_CAP,seed=SEED,output_dir=OUT)
print('Elapsed seconds:',round(time.perf_counter()-start,1))
display(res['audit']);print('Models evaluated:',res['metrics'].model.nunique(),'policy rows:',len(res['metrics']))
assert (pd.to_datetime(res['audit'].train_last_label,utc=True)<pd.to_datetime(res['audit'].validation_first_bar,utc=True)).all()

## Experiment 31 — Feature group ablation leaderboard: top 5, 10 and 20 positions

In [ ]:
leader=res['metrics'].groupby(['model','k'],as_index=False).agg(mean_return=('mean_daily','mean'),market_excess=('market_excess','mean'),win_rate=('win_rate','mean'),payoff=('payoff_ratio','mean'),profit_factor=('profit_factor','mean'),positive_folds=('mean_daily',lambda a:int((a>0).sum())))
for k in (5,10,20):
    print('Positions',k);display(leader.query('k==@k').sort_values('mean_return',ascending=False).round(6))
leader.to_csv(OUT/'leaderboard.csv',index=False)

## Experiment 32 — Per-fold performance, not only averages

In [ ]:
display(res['metrics'].query('k==5').pivot(index='model',columns='fold',values='mean_daily').round(6))

## Experiment 33 — Stable Ridge coefficient directions and importance

In [ ]:
w=res['weights'];summary=w.groupby(['feature','group'],as_index=False).agg(mean_abs=('absolute','mean'),mean_coef=('coefficient','mean'),positive_folds=('coefficient',lambda x:int((x>0).sum())))
summary['sign_consistent']=(summary.positive_folds==0)|(summary.positive_folds==3)
print('Top coefficients');display(summary.sort_values('mean_abs',ascending=False).head(40).round(7))
print('Importance by family');display(summary.groupby('group').agg(features=('feature','size'),mean_abs=('mean_abs','mean'),sign_stable=('sign_consistent','mean')).sort_values('mean_abs',ascending=False).round(6))
summary.to_csv(OUT/'feature_importance_summary.csv',index=False)

## Experiment 34 — Paired block-bootstrap uncertainty for all feature configurations

In [ ]:
paired=paired_block_intervals(res['daily'],k=5,reps=1000)
display(paired.groupby('model').agg(mean_delta=('mean_delta','mean'),positive_folds=('mean_delta',lambda x:int((x>0).sum())),significant_folds=('ci_low',lambda x:int((x>0).sum()))).sort_values('mean_delta',ascending=False).round(7))
paired.to_csv(OUT/'paired_block_intervals.csv',index=False)

## Experiment 35 — Familywise multiple-comparison reality check

In [ ]:
reality=reality_check(res['daily'],k=5,reps=1500,block=10)
display(reality.head(35).round(6));reality.to_csv(OUT/'reality_check.csv',index=False)
print('Familywise p values are exploratory and based on centered moving-block bootstrap, not an independent confirmation.')

## Experiment 36 — Do apparent winners survive removing unusually large outcomes?

In [ ]:
from stock_ml_dynamic.edge_discovery import tail_sensitivity
rows=[]
for name in ['ridge_all','only_market_breadth','only_momentum','without_market_breadth','top_30_coef']:
    z=res['trades'].query('model==@name and k==5')
    if len(z):
        for fold,g in z.groupby('fold'):
            t=tail_sensitivity(g.next_return);t['model']=name;t['fold']=fold;rows.append(t)
if rows:
    tails=pd.concat(rows,ignore_index=True)
    display(tails.groupby(['model','upper_tail_removed']).expectancy.mean().unstack().round(6))
    tails.to_csv(OUT/'tail_sensitivity.csv',index=False)

## Experiment 37 — Is the improvement a sector or market-beta effect?

In [ ]:
print('Market-breadth and momentum neutralization tests appear in the neutral_* rows above.')
print('True sector neutrality requires a point-in-time sector classification joined to each bar; we do NOT infer missing historical sectors.')
print('These neutralization tests are observational factor-proxy diagnostics, not complete risk-model hedges.')
print('Neutral variants:',res['metrics'].loc[res['metrics'].model.str.startswith('neutral_'),'model'].nunique())

## Experiment 38 — Final evidence table, open questions, and safeguards

In [ ]:
top=leader.query('k==5').sort_values('mean_return',ascending=False)
display(top.head(20).round(6))
print('Notebook 40 experiments are exploratory; do not select a deployment model from the maximum observed return alone.')
print('Development-only folds; final historical holdout CLOSED; prospective prediction ledger UNTOUCHED.')
print('Zero transaction costs is a research assumption. Close-to-close returns are not necessarily executable.')
print('Artifacts:',OUT)